<a id="curation-quickstart"></a>
# VideoDB Curation Quickstart: QA a Robot-Learning Dataset

Curation checks physical-AI training datasets (LeRobot v3, RLDS) against the format they declare and runs QA checks over the **decoded video**. The metadata is not taken on trust.

In this notebook you will:

1. **Build a small dataset yourself.** Cut three episodes from a public dataset, upload the clips to VideoDB, assemble them into a LeRobot v3 dataset and audit it.
2. **Connect a larger dataset in place.** Point Curation at a dataset that already lives in storage and audit every episode in batches. Nothing is copied.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/curation/quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### What the checks report

| Check | Question it answers |
|:--|:--|
| `completeness` | Does each camera hold exactly the frames the data table declares, at the declared timestamps? |
| `camera_validity` | Is every declared camera present, at its declared width and height, for the whole episode? |

Every episode and check gets one status. From worst to best: `error` (could not read or decode), `flagged` (measured, and something needs a human look), `insufficient` (could not decide; never a pass), `passed` and `not_applicable`.

A check reports evidence. It never drops, crops or rewrites your data; you decide what to keep.

<a id="install"></a>
## 1. Install and connect

Curation runs inside your VideoDB deployment. Set `VIDEO_DB_BASE_URL` (in your environment or `.env`) to a deployment with Curation enabled, and enter your API key when prompted.

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv pandas pyarrow requests

In [ ]:
import os
from getpass import getpass

from dotenv import load_dotenv
from videodb import connect

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")
base_url = os.getenv("VIDEO_DB_BASE_URL") or input("VideoDB endpoint (VIDEO_DB_BASE_URL): ")

conn = connect(api_key=api_key, base_url=base_url)
collection = conn.get_collection()
print("Connected to", base_url)
print("Collection:", collection.id)

<a id="helpers"></a>
## 2. Call the dataset endpoints

The Python SDK does not wrap the dataset endpoints yet. The connection already has `get`, `post` and `delete` methods that send your API key, so the notebook calls the endpoints through them.

Uploads and audits are asynchronous. They answer at once with `{id, status: "processing", output_url}`, and `wait_for` polls that `output_url` until the job finishes. Pass `callback_url` in either request to be called back instead of polling.

In [ ]:
import time

import pandas as pd
from IPython.display import display
from videodb.exceptions import InvalidRequestError


def error_detail(error):
    """The server's code, message and detail behind a failed call."""
    try:
        body = error.response.json()
    except Exception:
        return {"message": str(error)}
    return {k: body.get(k) for k in ("code", "message", "detail") if body.get(k) is not None}


def wait_for(job, timeout=3600, interval=5):
    """Poll an accepted upload or audit until it finishes; return its result."""
    deadline = time.monotonic() + timeout
    while True:
        body = conn.session.get(job["output_url"], timeout=30).json()
        if body.get("status") not in ("processing", "in progress"):
            break
        if time.monotonic() > deadline:
            raise TimeoutError(f"job {job['id']} still processing after {timeout}s")
        time.sleep(interval)
    response = body.get("response") or {}
    if not response.get("success"):
        raise RuntimeError(f"job {job['id']} failed: {response.get('message')} ({response.get('code')})")
    return response.get("data", {})


def paged(path, key, page_size=500, **params):
    """Read every item from a paged endpoint."""
    items, offset = [], 0
    while True:
        page = conn.get(path=path, params={**params, "offset": offset, "limit": page_size})
        items += page[key]
        offset += page_size
        if offset >= page["total"]:
            return items

---
# Part A: Build a small dataset yourself

<a id="source"></a>
## 3. Get a few source episodes

We need a few multi-camera robot recordings. [`lerobot/svla_so101_pickplace`](https://huggingface.co/datasets/lerobot/svla_so101_pickplace) (Apache-2.0) has 50 pick-and-place episodes from an SO-101 arm, filmed by two cameras at 640×480 and 30 fps.

LeRobot v3 packs many episodes into one video file per camera. We download the metadata and both camera files (about 85 MB), then cut three episodes into separate clips. Treat them as recordings from your own robot.

In [ ]:
import json
from pathlib import Path

import requests

SOURCE_REPO = "lerobot/svla_so101_pickplace"
SOURCE_URL = f"https://huggingface.co/datasets/{SOURCE_REPO}/resolve/main"
EPISODES = [0, 1, 2]

work = Path("curation_quickstart")
source_dir = work / "source"


def download(path):
    target = source_dir / path
    if not target.exists():
        target.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(f"{SOURCE_URL}/{path}", stream=True, timeout=60) as response:
            response.raise_for_status()
            with open(target, "wb") as handle:
                for chunk in response.iter_content(1 << 20):
                    handle.write(chunk)
    return target


info = json.loads(download("meta/info.json").read_text())
episodes = pd.read_parquet(download("meta/episodes/chunk-000/file-000.parquet"))
cameras = {key: f for key, f in info["features"].items() if f.get("dtype") == "video"}

print("Robot:", info["robot_type"], "| fps:", info["fps"], "| episodes:", info["total_episodes"])
for key, feature in cameras.items():
    height, width, _ = feature["shape"]
    print(f"Camera {key}: {width}x{height}")
display(episodes.loc[EPISODES, ["episode_index", "length", "tasks"]])

Each episode row gives a `from_timestamp` and `to_timestamp` inside each camera's file. Cut those intervals into one H.264 clip per episode and camera. FFmpeg comes preinstalled on Colab; elsewhere, install it first.

In [ ]:
import subprocess

clips_dir = work / "clips"
clips_dir.mkdir(parents=True, exist_ok=True)
clips = {}

for camera in cameras:
    row = episodes.iloc[0]
    video_file = download(info["video_path"].format(
        video_key=camera,
        chunk_index=int(row[f"videos/{camera}/chunk_index"]),
        file_index=int(row[f"videos/{camera}/file_index"]),
    ))
    for episode in EPISODES:
        row = episodes.loc[episode]
        start = row[f"videos/{camera}/from_timestamp"]
        frames = int(row["length"])
        clip = clips_dir / f"episode_{episode}_{camera.split('.')[-1]}.mp4"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-ss", f"{start:.6f}", "-i", str(video_file),
             "-frames:v", str(frames), "-an", "-c:v", "libx264", "-pix_fmt", "yuv420p",
             "-crf", "18", str(clip)],
            check=True,
        )
        clips[(episode, camera)] = clip
        print(f"episode {episode} {camera}: {frames} frames -> {clip.name}")

<a id="upload-videos"></a>
## 4. Upload the clips to VideoDB

A dataset is filled from videos already in VideoDB. Upload each clip and keep its video ID.

In [ ]:
video_ids = {}
for (episode, camera), clip in clips.items():
    video = collection.upload(file_path=str(clip), name=f"curation-quickstart {clip.stem}")
    video_ids[(episode, camera)] = video.id
    print(f"episode {episode} {camera}: {video.id}")

<a id="create"></a>
## 5. Create an empty dataset

Without a `source`, Curation creates an empty LeRobot v3 dataset in your deployment's storage. Declare its frame rate, robot, task and cameras. Each camera's width and height are the contract that `camera_validity` later checks against.

In [ ]:
camera_geometry = {
    key: {"width": feature["shape"][1], "height": feature["shape"][0]}
    for key, feature in cameras.items()
}

created = conn.post(path="dataset", data={
    "name": "SO-101 pick and place (quickstart)",
    "description": "Three episodes cut from lerobot/svla_so101_pickplace",
    "format": {
        "layout": "lerobot",
        "fps": info["fps"],
        "robot_type": info["robot_type"],
        "task": episodes.loc[EPISODES[0], "tasks"][0],
        "cameras": camera_geometry,
    },
})
created_id = created["id"]
print("Dataset:", created_id)
print(json.dumps(created, indent=2))

<a id="fill"></a>
## 6. Fill it, one video per episode and camera

Each upload copies one VideoDB video into an episode's camera slot. Curation decodes the video, writes the episode's frame table from the decoded timestamps and regenerates the dataset metadata. Uploads to one dataset run one at a time.

To see how QA reports a gap, we **leave out the side camera of the last episode**. Remove it from `SKIP` to build a complete dataset.

In [ ]:
SKIP = {(EPISODES[-1], "observation.images.side")}

for (episode, camera), video_id in video_ids.items():
    if (episode, camera) in SKIP:
        print(f"episode {episode} {camera}: skipped on purpose")
        continue
    job = conn.post(path=f"dataset/{created_id}/upload",
                    data={"video_id": video_id, "episode": episode, "camera": camera})
    result = wait_for(job)
    gaps = len(result.get("validation_errors", []))
    print(f"episode {episode} {camera}: {result['path']}"
          + (f"  ({gaps} validation issue(s) while incomplete)" if gaps else ""))

<a id="validate"></a>
## 7. Validate and list episodes

`validate` reads the dataset as LeRobot v3 and lists every way it breaks the layout. Because one camera is missing, expect a `VIDEO_BINDING_MISSING` problem.

In [ ]:
try:
    validation = conn.post(path=f"dataset/{created_id}/validate")
    print("Valid:", json.dumps(validation["summary"], indent=2))
except InvalidRequestError as error:
    print("Not valid:")
    print(json.dumps(error_detail(error), indent=2))

In [ ]:
listing = conn.get(path=f"dataset/{created_id}/episodes")
display(pd.DataFrame([
    {"episode": e["episode"], "frames": e["length"], "cameras": ", ".join(e["cameras"]),
     "missing": ", ".join(e.get("unbound_cameras", {}))}
    for e in listing["episodes"]
]))

<a id="audit"></a>
## 8. Audit the dataset

Start an audit of every episode with both checks. When it finishes, the result holds counts per check and status. Per-episode results are paged separately.

In [ ]:
audit_job = conn.post(path=f"dataset/{created_id}/audit",
                      data={"episodes": "all", "checks": ["completeness", "camera_validity"]})
audit = wait_for(audit_job)
print("Overall status:", audit["status"])
display(pd.DataFrame(audit["summary"]).T)

In [ ]:
def issue_codes(result):
    """Every issue code in one check's result, including per-camera issues."""
    codes = [i["code"] for i in result.get("issues", [])]
    for per_camera in (result.get("sequences") or result.get("cameras") or {}).values():
        codes += [i["code"] for i in per_camera.get("issues", [])]
    return sorted(set(codes))


def audit_table(dataset_id, job_id):
    rows = paged(f"dataset/{dataset_id}/audit/{job_id}/episodes", "episodes")
    table = pd.DataFrame([
        {"episode": r["episode"], "status": r["status"],
         **{check: result["status"] for check, result in r["checks"].items()},
         "issues": ", ".join(sorted({c for result in r["checks"].values() for c in issue_codes(result)}))}
        for r in rows
    ])
    return table, rows


created_table, created_rows = audit_table(created_id, audit_job["id"])
display(created_table)

Drill into anything that did not pass. Each issue carries a code, a message, a status and its evidence: the observed and expected values, and where they differ.

In [ ]:
def explain(rows):
    for row in rows:
        if row["status"] == "passed":
            continue
        print(f"Episode {row['episode']}: {row['status']}")
        for check, result in row["checks"].items():
            found = [(None, i) for i in result.get("issues", [])]
            for camera, per_camera in (result.get("sequences") or result.get("cameras") or {}).items():
                found += [(camera, i) for i in per_camera.get("issues", [])]
            for camera, i in found:
                where = f" [{camera}]" if camera else ""
                print(f"  {check}{where}: {i['code']} ({i['status']}): {i['message']}")


explain(created_rows)

The missing side camera shows up in both checks. `camera_validity` flags `MISSING_CAMERA`. `completeness` reports `BINDING_UNESTABLISHED` (`insufficient`), because a camera with no video cannot be compared with the data table. The episodes with both cameras pass both checks: every decoded frame lands where the data table says it should.

---
# Part B: Connect a larger dataset in place

<a id="connect"></a>
## 9. Connect an existing dataset

Give a `source` and Curation reads the dataset where it lives, over HTTPS. It never copies or writes it. The source can be a Hugging Face revision, or an Azure Blob prefix: for a private container, store a read-only SAS in your deployment's Key Vault and pass `credential: {kind: "key_vault_secret", name: ...}`. Never put a SAS in the URL.

`layout: "auto"` detects LeRobot v3 or RLDS. To test a larger dataset, change `HF_REPO`:

| Dataset | Episodes | Cameras |
|:--|--:|:--|
| `lerobot/svla_so101_pickplace` | 50 | 2 × 640×480 |
| `lerobot/koch_pick_place_1_lego` | 102 | 2 × 640×480 |
| `lerobot/pusht` | 206 | 1 × 96×96 |
| `lerobot/xarm_lift_medium` | 800 | 1 × 84×84 |

In [ ]:
HF_REPO = "lerobot/svla_so101_pickplace"

connected = conn.post(path="dataset", data={
    "name": HF_REPO,
    "source": {"url": f"https://huggingface.co/datasets/{HF_REPO}/resolve/main"},
    "format": {"layout": "auto"},
})
connected_id = connected["id"]
print("Dataset:", connected_id, "| layout:", connected["layout"], "| managed:", connected["managed"])
print(json.dumps(connected["summary"], indent=2))

<a id="batches"></a>
## 10. Audit every episode, in batches

One audit covers at most 50 episodes, so a larger dataset is audited in batches. Every batch is submitted up front; they queue on the deployment's worker pool, and the cell then waits for each. Lower `MAX_EPISODES` for a quicker run.

In [ ]:
BATCH_SIZE = 50
MAX_EPISODES = None  # e.g. 100 to audit only the first 100 episodes

indices = [e["episode"] for e in paged(f"dataset/{connected_id}/episodes", "episodes")]
indices = indices[:MAX_EPISODES]
batches = [indices[i:i + BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]

jobs = [conn.post(path=f"dataset/{connected_id}/audit", data={"episodes": batch}) for batch in batches]
print(f"Submitted {len(jobs)} audit(s) covering {len(indices)} episodes")

tables, connected_rows = [], []
for number, job in enumerate(jobs, 1):
    started = time.monotonic()
    result = wait_for(job)
    table, rows = audit_table(connected_id, job["id"])
    tables.append(table)
    connected_rows += rows
    print(f"batch {number}/{len(jobs)}: {result['status']} ({time.monotonic() - started:.0f}s)")

connected_table = pd.concat(tables, ignore_index=True)

In [ ]:
print("Episodes by status, per check")
display(pd.DataFrame({
    check: connected_table[check].value_counts() for check in ("completeness", "camera_validity")
}).fillna(0).astype(int))

print("Most common issues")
display(connected_table["issues"].str.split(", ").explode().loc[lambda s: s != ""].value_counts().to_frame("episodes"))

print("Episodes that need a look")
display(connected_table[connected_table["status"] != "passed"])

In [ ]:
# Detail for the first few episodes that did not pass
explain([r for r in connected_rows if r["status"] != "passed"][:5])

<a id="use"></a>
## 11. Use the results

Curation never removes episodes. Decide what to train on from the audit, for example only episodes that passed every check:

In [ ]:
keep = connected_table.loc[connected_table["status"] == "passed", "episode"].tolist()
review = connected_table.loc[connected_table["status"] != "passed", "episode"].tolist()
print(f"{len(keep)} episodes pass every check; {len(review)} need review: {review[:20]}")

<a id="cleanup"></a>
## 12. Clean up

Deleting a connected dataset only disconnects it; its files are never touched. Deleting a created dataset removes its files from your deployment's storage.

In [ ]:
for dataset_id in (created_id, connected_id):
    conn.delete(path=f"dataset/{dataset_id}")
    print("Deleted dataset", dataset_id)

for video_id in video_ids.values():
    collection.delete_video(video_id)
print(f"Deleted {len(video_ids)} uploaded clips")